# FactorCalculator 使用示例

本 Notebook 演示 `factor_calculator` 包的各种用法，包括解析单元规格、创建实例、以及使用计算器进行因子计算。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

## 1. 解析单元规格（Unit Specifications）

In [2]:
from factor_calculator import parse_unit_spec

specs = [
    "KlineDMU(45)",
    "KlineDMU(interval=5)",
    "BiquotePEU(watching_time=60)",
    "PositionPnlDMU",
]

for spec in specs:
    class_name, params = parse_unit_spec(spec)
    print(f"{spec!r}")
    print(f"  -> class: {class_name}, params: {params!r}")

'KlineDMU(45)'
  -> class: KlineDMU, params: '45'
'KlineDMU(interval=5)'
  -> class: KlineDMU, params: 'interval=5'
'BiquotePEU(watching_time=60)'
  -> class: BiquotePEU, params: 'watching_time=60'
'PositionPnlDMU'
  -> class: PositionPnlDMU, params: ''


## 2. 列出可用的 DMU / PEU 类

> 需要安装 `rbt` 包才能运行此部分。

In [3]:
from factor_calculator import get_available_classes

print("所有可用单元:")
for cls in get_available_classes():
    print(f"  - {cls}")

print("\nDMU 类:")
for cls in get_available_classes("DMU"):
    print(f"  - {cls}")

print("\nPEU 类:")
for cls in get_available_classes("PEU"):
    print(f"  - {cls}")

所有可用单元:
  - BiquoteClosePEU
  - BiquotePEU
  - BiquoteStopClosePEU
  - BtsSimplePEU
  - FixedHoldingPEU
  - KlineDMU
  - MdDMU
  - MidPositionPnlDMU
  - MoIntentionDMU
  - OlsTrendDMU
  - PassThroughDMU
  - PositionGenDMU
  - PositionPnlDMU
  - SimpleBiquotePEU
  - SpreadDMU
  - TimePeriodDMU
  - TrendDMU

DMU 类:
  - KlineDMU
  - MdDMU
  - MidPositionPnlDMU
  - MoIntentionDMU
  - OlsTrendDMU
  - PassThroughDMU
  - PositionGenDMU
  - PositionPnlDMU
  - SpreadDMU
  - TimePeriodDMU
  - TrendDMU

PEU 类:
  - BiquoteClosePEU
  - BiquotePEU
  - BiquoteStopClosePEU
  - BtsSimplePEU
  - FixedHoldingPEU
  - SimpleBiquotePEU


## 3. 创建单元实例

> 需要安装 `rbt` 包才能运行此部分。

In [4]:
from factor_calculator import create_unit, create_units

# 创建单个单元
unit = create_unit("KlineDMU(interval=5)")
print(f"Created: {unit}")
print(f"  Name: {unit.name}")
print(f"  Interval: {unit.interval}")

Created: <rbt.dmu.kline_dmu.KlineDMU object at 0x1059cb580>
  Name: KlineDMU_v1_5min
  Interval: 5


In [5]:
# 批量创建多个单元
units = create_units(["PositionPnlDMU", "FixedHoldingPEU(watching_mds=100)", "TrendDMU"])
print(f"创建了 {len(units)} 个单元:")
for u in units:
    print(f"  - {u.name}")

创建了 3 个单元:
  - PositionPnlDMU_v0
  - FixedHoldingPEU_v1_100t
  - TrendDMU_v0_90


## 4. FactorCalculator（完整集成）

完整版计算器，集成 RBT Strategy 引擎、行情加载、结果数据库。

In [6]:
from factor_calculator import FactorCalculator
import datetime

#初始化（root_path 替代了旧的 db_directory，新增 frequency 参数）
calc = FactorCalculator()

#执行因子计算
result = calc.calculate(
    units=[
        "KlineDMU(interval=5)",
        # "BiquotePEU(1)",
        # "FixedHoldingPEU(10)",
    ],
    # load_factors=["KlineDMU__open", "KlineDMU__close"],
    contract="TL2603",
    trade_date=datetime.date(2026, 1, 6),
    frequency="tick",
)
result

TypeError: FactorCalculator.calculate() missing 1 required positional argument: 'load_factors'

## 5. CLI 命令行用法

安装包后可直接在终端使用 `factor-calculator` 命令：

```bash
# 列出可用单元
factor-calculator list
factor-calculator list --dmu
factor-calculator list --peu

# 计算因子
factor-calculator calculate \
    --db /path/to/results \
    --md /path/to/market/data \
    --units "KlineDMU(5),BiquotePEU(60)" \
    --contract IF2403 \
    --date 2024-03-15

# 查看已有因子
factor-calculator factors \
    --db /path/to/results \
    --contract IF2403 \
    --date 2024-03-15
```